In [2]:
import numpy as np
import pandas as pd
import os
import kagglehub
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

In [3]:
path = kagglehub.dataset_download("algozee/teenager-menthal-healy")

In [4]:
print('Arquivos do Dataset: ', os.listdir(path))

Arquivos do Dataset:  ['Teen_Mental_Health_Dataset.csv']


In [5]:
rota = os.path.join(path, 'Teen_Mental_Health_Dataset.csv')
df = pd.read_csv(rota)
df.shape

(1200, 13)

In [6]:
df.head()

,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,male,7.9,Instagram,7.4,2.9,3.01,1.5,low,2,2,1,0
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,0
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,0
3,15,male,7.4,TikTok,6.9,1.6,3.48,0.8,medium,1,7,9,0
4,15,female,4.7,Both,4.9,3.0,2.37,1.4,medium,3,5,2,0


In [7]:
#Converter as strings em números
interaction_map = {'low': 0, 'medium': 1, 'high': 2}
df['social_interaction_level'] = (
    df['social_interaction_level'].str.strip().str.lower().map(interaction_map)
)

le = LabelEncoder()
df['gender'] = le.fit_transform(df['gender'].str.strip())
df['platform_usage'] = le.fit_transform(df['platform_usage'].str.strip())

In [8]:
df.head()

,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,1,7.9,1,7.4,2.9,3.01,1.5,0,2,2,1,0
1,19,0,1.9,2,8.0,2.9,3.22,0.8,2,8,1,10,0
2,17,0,1.3,1,7.6,0.5,3.92,0.0,2,2,4,2,0
3,15,1,7.4,2,6.9,1.6,3.48,0.8,1,1,7,9,0
4,15,0,4.7,0,4.9,3.0,2.37,1.4,1,3,5,2,0


In [9]:
#8 horas de sono pois é saudável + 0 que garante que a diferença não resulte em negavito (8-10 = -2)
df['deficit_sono'] = np.maximum(0, 8 - df['sleep_hours'])
#o 1e-9 garante que caso alguém tenha colocado 0 horas de sono, não de erro, e divida por um número mínimo, como 0.0000000001
df['relacao_tela_sono'] = df['screen_time_before_sleep'] / (df['sleep_hours'] + 1e-9)
#fazer a média dos pontos negativos
df['pontuacao_carga_mental'] = (
    df['stress_level'] + df['anxiety_level'] + df['addiction_level']
) / 3

In [10]:
df.head()

,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label,deficit_sono,relacao_tela_sono,pontuacao_carga_mental
0,14,1,7.9,1,7.4,2.9,3.01,1.5,0,2,2,1,0,0.6,0.391892,1.666667
1,19,0,1.9,2,8.0,2.9,3.22,0.8,2,8,1,10,0,0.0,0.362500,6.333333
2,17,0,1.3,1,7.6,0.5,3.92,0.0,2,2,4,2,0,0.4,0.065789,2.666667
3,15,1,7.4,2,6.9,1.6,3.48,0.8,1,1,7,9,0,1.1,0.231884,5.666667
4,15,0,4.7,0,4.9,3.0,2.37,1.4,1,3,5,2,0,3.1,0.612245,3.333333


In [13]:
X = df.drop('depression_label', axis=1)
y = df['depression_label']

print(f'Formato de X: {X.shape}')
print(f'Colunas: {X.columns.tolist()}\n')


Formato de X: (1200, 15)
Colunas: ['age', 'gender', 'daily_social_media_hours', 'platform_usage', 'sleep_hours', 'screen_time_before_sleep', 'academic_performance', 'physical_activity', 'social_interaction_level', 'stress_level', 'anxiety_level', 'addiction_level', 'deficit_sono', 'relacao_tela_sono', 'pontuacao_carga_mental']



### PADRONIZAR OS NÚMEROS

**Fórmula matemática para o `fit_transform(X)`**

$$z = \frac{x - μ}{σ}$$

Ela pega cada valor (x), subtrai a média daquela coluna (μ) e divide pelo desvio padrão (σ).

In [18]:
nivelar = StandardScaler()
X_nivelado = nivelar.fit_transform(X)

print('Média após nivelar: ', X_nivelado.mean(axis=0).round(3))
print('Desvio Padrão após nivelar: ', X_nivelado.std(axis=0).round(3))

Média após nivelar:  [ 0.  0. -0.  0. -0.  0. -0. -0.  0. -0. -0. -0.  0.  0.  0.]
Desvio Padrão após nivelar:  [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


### PCA (Principal Component Analysis) -> Variância Explicada

Como o dataset tem muitas colunas, essa técnica vai reduzir para `super colunas`, um processo de `Cluster`, agrupando por semelhança

In [25]:
reduz_dados = PCA()
reduz_dados.fit(X_nivelado)

#explained_variance -> diz quanto cada super coluna capturou da variancia da tabela toda
#vai pegar o valor de explained... e vai somar uma por uma: 1 super coluna, depois 1 + 2, depois 1+2+3...n
variancia = np.cumsum(reduz_dados.explained_variance_ratio_)

print("\nComponentes | Variância individual | Variância acumulada")
print("  " + "-" * 55)
for i, (individual, acumulada) in enumerate(
    zip(reduz_dados.explained_variance_ratio_, variancia), 1):
        mark = " ← escolhido" if i == 2 else (" ← 80%" if abs(acumulada - 0.80) < 0.03 else "")
        print(f"PC{i:2d}        |  {individual:.4f}  ({individual*100:.1f}%)       |  {acumulada:.4f}  ({acumulada*100:.1f}%) {mark}")
        if i == 10:
            break



Componentes | Variância individual | Variância acumulada
  -------------------------------------------------------
PC 1        |  0.1654  (16.5%)       |  0.1654  (16.5%) 
PC 2        |  0.1342  (13.4%)       |  0.2996  (30.0%)  ← escolhido
PC 3        |  0.1027  (10.3%)       |  0.4023  (40.2%) 
PC 4        |  0.0726  (7.3%)       |  0.4749  (47.5%) 
PC 5        |  0.0716  (7.2%)       |  0.5464  (54.6%) 
PC 6        |  0.0696  (7.0%)       |  0.6160  (61.6%) 
PC 7        |  0.0680  (6.8%)       |  0.6840  (68.4%) 
PC 8        |  0.0677  (6.8%)       |  0.7517  (75.2%) 
PC 9        |  0.0646  (6.5%)       |  0.8162  (81.6%)  ← 80%
PC10        |  0.0627  (6.3%)       |  0.8790  (87.9%) 


vamos usar 30% devido trabalhar com 2 dimensões, ficando mais fácil de enxergar

In [28]:
reduz_dados2 = PCA(n_components=2, random_state=42)
X_pca = reduz_dados2.fit_transform(X_nivelado)
X_pca.shape

(1200, 2)

In [39]:
nomes = X.columns.tolist()
receita_pca = pd.DataFrame(
    reduz_dados2.components_.T,
    index=nomes,
    columns=['PC1', 'PC2']
)

#sinal serve para dizer o quanto da determianda coluna afeta a super coluna
#o PC1 aumenta se o deficit de sono aumenta, assim como o PC1 diminui se as horas de sono aumentam
print("Como o PCA é Calculado — PC1 (Eixo do Sono):")
top_pc1 = receita_pca["PC1"].abs().sort_values(ascending=False).head(5)
for feat, val in top_pc1.items():
    sinal = "+" if receita_pca.loc[feat, "PC1"] > 0 else "-"
    print(f"{sinal} {feat:35s} |loading| = {val:.3f}")

Como o PCA é Calculado — PC1 (Eixo do Sono):
+ deficit_sono                        |loading| = 0.537
- sleep_hours                         |loading| = 0.535
+ relacao_tela_sono                   |loading| = 0.526
+ screen_time_before_sleep            |loading| = 0.300
+ pontuacao_carga_mental              |loading| = 0.160


In [38]:
print("Como o PCA é Calculado — PC2 (Eixo da Carga Mental):")
top_pc2 = receita_pca["PC2"].abs().sort_values(ascending=False).head(5)
for feat, val in top_pc2.items():
    sinal = "+" if receita_pca.loc[feat, "PC2"] > 0 else "-"
    print(f"{sinal} {feat:35s} |loading| = {val:.3f}")

Como o PCA é Calculado — PC2 (Eixo da Carga Mental):
+ pontuacao_carga_mental              |loading| = 0.681
+ anxiety_level                       |loading| = 0.417
+ stress_level                        |loading| = 0.398
+ addiction_level                     |loading| = 0.382
- relacao_tela_sono                   |loading| = 0.143


In [40]:
print("\nPCA 2D: variância explicada por eixo")
print(f"PC1: {reduz_dados2.explained_variance_ratio_[0]*100:.1f}% -> Eixo do Sono")
print(f"PC2: {reduz_dados2.explained_variance_ratio_[1]*100:.1f}% -> Eixo da Carga Mental")
print(f"Total: {sum(reduz_dados2.explained_variance_ratio_)*100:.1f}%\n")


PCA 2D: variância explicada por eixo
PC1: 16.5% -> Eixo do Sono
PC2: 13.4% -> Eixo da Carga Mental
Total: 30.0%



### K-Means